# Collect the data to reproduce Figure 6 Clustering Results
Sink levels (lstart, lifetime) in different models.

You need two files:
- `file_path`: model generations from `./outputs/aime24/vllm/`.
- `sink_info_path`: sink detection results from `./src/hidden_state_base.py`.


# Reproduce Clustering Results
This script collects intermediate hidden states, MLP outputs, and attention outputs from specific layers.
We used deepseek-14b as example for the clustering analysis in the paper.
- Set the parameters below. To reproduce the paper results, use `n_gen=10`, `target_layer_idx=17` to `22`, and `sample_num=10`.
- Collecting one layer takes about 6 minutes and uses approximately 4.8 GB of storage.
- To reduce CPU RAM usage, this script processes one layer at a time. Run it multiple times to collect all target layers, or change `target_layer_idx` to analyze different layers.

Output files:
```text
./results/example/attn_output_residual_mlp_pt/creation_layer_info_dict_deepseek-14b_n_{n_gen}_other_{target_layer_idx}.pt
```

This can be quite large, so make sure you have enough storage space before running the script.

The collected data will be used for clustering analysis to understand the behavior of secondary attention sinks in the model using `03.b.clustering_analysis.ipynb`.

In [1]:
import json
import matplotlib.pyplot as plt
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.utils import (
    compute_matrix_based_entropy,
    get_full_text,
    get_full_text_chat,
)

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="auto",
    attn_implementation="sdpa",
    cache_dir="/data/models",
)
model.eval()

/home/thw20/.conda/envs/sink/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [01:20<00:00, 20.07s/it]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 5120)
    (layers): ModuleList(
      (0-47): 48 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=5120, out_features=5120, bias=True)
          (k_proj): Linear(in_features=5120, out_features=1024, bias=True)
          (v_proj): Linear(in_features=5120, out_features=1024, bias=True)
          (o_proj): Linear(in_features=5120, out_features=5120, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=5120, out_features=13824, bias=False)
          (up_proj): Linear(in_features=5120, out_features=13824, bias=False)
          (down_proj): Linear(in_features=13824, out_features=5120, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((5120,), eps=1e-05)
        (post_attention_layernorm): Qwen2RMSNorm((5120,), eps=1e-05)
      )
    )
    (norm): Qwen2RMSNorm((5120,), eps=1e-05)
    (

In [5]:
from transformers.models.qwen2.modeling_qwen2 import FlashAttentionKwargs, Cache, Unpack, apply_rotary_pos_emb, eager_attention_forward, ALL_ATTENTION_FUNCTIONS
from typing import Optional, Callable
from collections import defaultdict


def patch_qwen_attention(self, collector):
    def patched_forward(
        hidden_states: torch.Tensor,
        position_embeddings: tuple[torch.Tensor, torch.Tensor],
        attention_mask: Optional[torch.Tensor],
        past_key_value: Optional[Cache] = None,
        cache_position: Optional[torch.LongTensor] = None,
        collect_target: Optional[str] = None,
        **kwargs: Unpack[FlashAttentionKwargs],
    ) -> tuple[torch.Tensor, Optional[torch.Tensor], Optional[tuple[torch.Tensor]]]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.head_dim)

        query_states = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        key_states = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        cos, sin = position_embeddings
        if "q" in collect_target:
            collector["q"].append(query_states.detach().float().cpu())
        if "k" in collect_target:
            collector["k"].append(key_states.detach().float().cpu())
        if "v" in collect_target:
            collector["v"].append(value_states.detach().float().cpu())
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)
        if "cos" in collect_target:
            collector["cos"] = cos.detach().float().cpu()
        if "sin" in collect_target:
            collector["sin"] = sin.detach().float().cpu()
        if "roped_q" in collect_target:
            collector["roped_q"].append(query_states.detach().float().cpu())
        if "roped_k" in collect_target:
            collector["roped_k"].append(key_states.detach().float().cpu())
        
        if past_key_value is not None:
            # sin and cos are specific to RoPE models; cache_position needed for the static cache
            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}
            key_states, value_states = past_key_value.update(key_states, value_states, self.layer_idx, cache_kwargs)

        attention_interface: Callable = eager_attention_forward
        if self.config._attn_implementation != "eager":
            attention_interface = ALL_ATTENTION_FUNCTIONS[self.config._attn_implementation]

        if "attn_weights" in collect_target:
            attention_interface = eager_attention_forward
        else:
            attention_interface = ALL_ATTENTION_FUNCTIONS["sdpa"]

        attn_output, attn_weights = attention_interface(
            self,
            query_states,
            key_states,
            value_states,
            attention_mask,
            dropout=0.0 if not self.training else self.attention_dropout,
            scaling=self.scaling,
            sliding_window=self.sliding_window,  # main diff with Llama
            **kwargs,
        )
        if "attn_weights" in collect_target:
            collector["attn_weights"].append(attn_weights.detach().float().cpu())
        del attn_weights
        attn_weights = None

        attn_output = attn_output.reshape(*input_shape, -1).contiguous()
        attn_output = self.o_proj(attn_output)
        if "attn_output" in collect_target:
            collector["attn_output"].append(attn_output.detach().float().cpu())
        return attn_output, attn_weights
    self.forward = patched_forward

def patch_qwen_mlp(self, collector):
    # ['mlp_input', 'up', 'gate', 'act', 'elementwise', 'down']
    def patched_forward(x, collect_target = ['mlp_input', 'down']):
        up = self.up_proj(x)
        gate = self.gate_proj(x)
        act = self.act_fn(gate)
        elementwise = act * up
        down = self.down_proj(elementwise)
        if "mlp_input" in collect_target:
            collector["mlp_input"].append(x.detach().float().cpu())
        if "up" in collect_target:
            collector["up"].append(up.detach().float().cpu())
        if "gate" in collect_target:
            collector["gate"].append(gate.detach().float().cpu())
        if "act" in collect_target:
            collector["act"].append(act.detach().float().cpu())
        if "elementwise" in collect_target:
            collector["elementwise"].append(elementwise.detach().float().cpu())
        if "down" in collect_target:
            collector["down"].append(down.detach().float().cpu())
        return down
    self.forward = patched_forward

# collect attention output 
from collections import defaultdict
def attn_output_hook(self, input, output, collector):
    attn_output, attn_weights = output
    collector['attn_output'].append(attn_output.detach().cpu())
    return output

def mlp_input_hook(self, input, output, collector):
    hidden_states = input[0]
    collector['mlp_input'].append(hidden_states.detach().cpu())
    collector['down'].append(output.detach().cpu())
    return output



# Change the parameters below to reproduce the paper results.

In [ ]:
# Set parameters
from tqdm import tqdm
use_chat_template = True
sample_num = 10
n_gen = 10
target_layer_idx = 25

cosine_similarities_dict = {}
sink_info_path = f"./results/sink_detection/deepseek-14b/sink_detection_n_{n_gen}_deepseek-14b.jsonl"
file_path = f"./outputs/aime24/vllm/output_n_{n_gen}_deepseek-14b.jsonl"

print("=" * 50)
print(f"Sink info path: {sink_info_path}")
print(f"File path: {file_path}")
print(f"Target layer index: {target_layer_idx}")
print("=" * 50)

In [ ]:
collector = defaultdict(list)
for idx, layer in enumerate(model.model.layers):
    if idx != target_layer_idx:
        continue
    layer.self_attn.register_forward_hook(
        lambda module, input, output, c=collector: attn_output_hook(module, input, output, c)
    )
    layer.mlp.register_forward_hook(
        lambda module, input, output, c=collector: mlp_input_hook(module, input, output, c)
    )

cliff_token_collector = []
with open(file_path, 'r') as f:
    lines = f.readlines()
    lines = [json.loads(line) for line in lines]
    sample_num = min(len(lines), sample_num)

for sample_index in tqdm(range(sample_num)):
    # if sample_index != args.sample_index:
    #     continue
    sample = lines[sample_index]
    prompt = sample['prompt']
    response = sample['output']
    full_text = get_full_text(prompt, response, model_name) if not use_chat_template else get_full_text_chat(prompt, response, model_name, tokenizer)

    tokenized_ids = tokenizer(
        [full_text],
        return_tensors="pt",
        add_special_tokens=(not use_chat_template),
    ).to("cuda")

    # if "32" in model_short_name:
    #     tokenized_ids = {k: v[:, :9000] for k, v in tokenized_ids.items()}
    #     warning("token length is capped to 9000 for testing.")
    
    # locate target layers from sink info
    sink_pos_creation_life = set()
    target_layers = None
    with open(sink_info_path, 'r') as f:
        sink_info_lines = f.readlines()
        for sink_line in sink_info_lines:
            sink_data = json.loads(sink_line)
            if sink_data['sample_index'] == sample_index:
                candidate_sinks = sink_data['candidate_sinks']
                for pos, info in candidate_sinks.items():
                    # Extract the creation layers of the sinks
                    sink_pos_creation_life.add((int(pos), info['layer'][0], len(info['layer'])))

    # we are only interested in cliffs currently
    print(f"Identified sink positions and their creation layers and life levels: {sink_pos_creation_life}")
    target_layers = sorted(list(set(layer for _, layer, _ in sink_pos_creation_life)))
    print(f"Target layers for heavy patching: {target_layers}")

    with torch.no_grad():
        model_output = model(**tokenized_ids, output_hidden_states=True, return_dict=True, use_cache=False, collect_target=['attn_output'])
        collector['hidden_states'].append(model_output.hidden_states[target_layer_idx].detach().cpu())
    torch.cuda.empty_cache()



./results/sink_detection/deepseek-14b/sink_detection_n_10_deepseek-14b.jsonl
./outputs/aime24/vllm/output_n_10_deepseek-14b.jsonl


  0%|          | 0/10 [00:00<?, ?it/s]

Identified sink positions and their creation layers and life levels: {(2, 4, 43), (1, 4, 43), (0, 4, 43), (1898, 22, 22)}
Target layers for heavy patching: [4, 22]


 10%|█         | 1/10 [00:00<00:04,  2.08it/s]

Identified sink positions and their creation layers and life levels: {(2, 4, 43), (1, 4, 43), (1847, 22, 22), (0, 4, 43)}
Target layers for heavy patching: [4, 22]


...

 80%|████████  | 8/10 [00:03<00:00,  2.24it/s]

Identified sink positions and their creation layers and life levels: {(2, 4, 43), (1, 4, 43), (0, 4, 43), (1409, 22, 22)}
Target layers for heavy patching: [4, 22]


 90%|█████████ | 9/10 [00:03<00:00,  2.63it/s]

Identified sink positions and their creation layers and life levels: {(2, 4, 43), (1, 4, 43), (0, 4, 43)}
Target layers for heavy patching: [4]


100%|██████████| 10/10 [00:04<00:00,  2.36it/s]


In [ ]:
collector.keys(), len(collector['hidden_states']),len(collector['mlp_input']), model_output.hidden_states[target_layer_idx].shape, collector['mlp_input'][-1].shape

(dict_keys(['attn_output', 'mlp_input', 'down', 'hidden_states']),
 10,
 10,
 torch.Size([1, 3324, 5120]),
 torch.Size([1, 3324, 5120]))

In [10]:
# save mlp_input, attention_output, bos_down
collector.keys() # check collected keys

len(collector['mlp_input']) # sample length

with open(sink_info_path, 'r') as f:
    sink_info_lines = f.readlines()

# NOTE: Cliff just means the identified secondary attention sinks, and other means the tokens with same token ids as cliff but not identified as sinks. We want to compare the difference between these two groups of tokens.
save_dict = {
    'bos': {
        "down": collector['down'][0].squeeze()[0],
        "mlp_input": collector['mlp_input'][0].squeeze()[0],
        "attn_output": collector['attn_output'][0].squeeze()[0],
        "residual": collector['hidden_states'][0].squeeze()[0],
    },
    "cliff": {
        "down": [],
        "mlp_input": [],
        "attn_output": [],
        "residual": [],
        "token_id": [],
        "sample_index": [],
        'position': []
    },
    "other": {
        "down": [],
        "mlp_input": [],
        "attn_output": [],
        "residual": [],
        "token_id": [],
        "sample_index": [],
        'position': []
    }
}

unique_sample_id_token_ids_pos = set()

for sample_index in range(sample_num):
    sink_line = sink_info_lines[sample_index]
    sink_data = json.loads(sink_line)
    if sink_data['sample_index'] != sample_index:
        raise ValueError("Sample index mismatch.")
    candidate_sinks = sink_data['candidate_sinks']

    exclude_positions = [0, 1, 2]
    pos_list = [int(k) for k in candidate_sinks.keys() if int(k) not in exclude_positions]
    token_id_list = [info['token_id'] for pos, info in candidate_sinks.items() if int(pos) not in exclude_positions]
    save_dict["cliff"]['token_id'] += token_id_list
    save_dict["cliff"]["sample_index"] += [sample_index] * len(token_id_list)
    save_dict["cliff"]['position'] += pos_list
    save_dict["cliff"]["down"].append(collector['down'][sample_index].squeeze()[pos_list])
    save_dict["cliff"]["mlp_input"].append(collector['mlp_input'][sample_index].squeeze()[pos_list])
    save_dict["cliff"]["attn_output"].append(collector['attn_output'][sample_index].squeeze()[pos_list])
    save_dict["cliff"]["residual"].append(collector['hidden_states'][sample_index].squeeze()[pos_list])


    # add other token with same token_ids 
    prompt, response = lines[sample_index]['prompt'], lines[sample_index]['output']
    full_text = get_full_text(prompt, response, model_name) if not use_chat_template else get_full_text_chat(prompt, response, model_name, tokenizer)

    tokenized_ids = tokenizer(
        [full_text],
        # padding="longest",
        return_tensors="pt",
        add_special_tokens=(not use_chat_template),
    ).to("cuda")

    same_token_id_other_data_pos_list = [pos for pos, tid in enumerate(tokenized_ids['input_ids'][0]) if (tid in token_id_list) & (pos not in pos_list)]
    same_token_id_other_data_token_id_list = tokenized_ids['input_ids'][0][same_token_id_other_data_pos_list]
    save_dict["other"]['token_id'] += same_token_id_other_data_token_id_list
    save_dict["other"]["sample_index"] += [sample_index] * len(same_token_id_other_data_pos_list)
    save_dict["other"]['position'] += same_token_id_other_data_pos_list
    save_dict["other"]["down"].append(collector['down'][sample_index].squeeze()[same_token_id_other_data_pos_list])
    save_dict["other"]["mlp_input"].append(collector['mlp_input'][sample_index].squeeze()[same_token_id_other_data_pos_list])
    save_dict["other"]["attn_output"].append(collector['attn_output'][sample_index].squeeze()[same_token_id_other_data_pos_list])  
    save_dict["other"]["residual"].append(collector['hidden_states'][sample_index].squeeze()[same_token_id_other_data_pos_list])




save_dict["cliff"]["down"] = torch.cat(save_dict["cliff"]["down"], dim=0)
save_dict["cliff"]["mlp_input"] = torch.cat(save_dict["cliff"]["mlp_input"], dim=0)
save_dict["cliff"]["attn_output"] = torch.cat(save_dict["cliff"]["attn_output"], dim=0)
save_dict["cliff"]["residual"] = torch.cat(save_dict["cliff"]["residual"], dim=0)
save_dict["other"]["down"] = torch.cat(save_dict["other"]["down"], dim=0)
save_dict["other"]["mlp_input"] = torch.cat(save_dict["other"]["mlp_input"], dim=0)
save_dict["other"]["attn_output"] = torch.cat(save_dict["other"]["attn_output"], dim=0)
save_dict["other"]["residual"] = torch.cat(save_dict["other"]["residual"], dim=0)

print(f"collected {len(save_dict['cliff']['attn_output'])} cliff")
print(f"collected {len(save_dict['other']['attn_output'])} other")

# save_dict_path = f"creation_layer_info_dict_deepseek-14b_n_{n_gen}.pt"
# torch.save(save_dict, save_dict_path)

unique_sample_id_token_ids_pos = set()
filtered_data = {
    'down': [],
    'mlp_input': [],
    'attn_output': [],
    'residual': [],
    'token_id': [],
    'sample_index': [],
    'position': []
}

for i in range(len(save_dict['cliff']['token_id'])):
    token_id = save_dict['cliff']['token_id'][i]
    sample_index = save_dict['cliff']['sample_index'][i]
    position = save_dict['cliff']['position'][i]

    key_tuple = (token_id, position)
    if key_tuple in unique_sample_id_token_ids_pos:
        continue

    unique_sample_id_token_ids_pos.add(key_tuple)

    # Collect only the first occurrence of each unique combination
    filtered_data['token_id'].append(token_id)
    filtered_data['sample_index'].append(sample_index)
    filtered_data['position'].append(position)
    filtered_data['down'].append(save_dict['cliff']['down'][i])
    filtered_data['mlp_input'].append(save_dict['cliff']['mlp_input'][i])
    filtered_data['attn_output'].append(save_dict['cliff']['attn_output'][i])
    filtered_data['residual'].append(save_dict['cliff']['residual'][i])

filtered_data['down'] = torch.stack(filtered_data['down'])
filtered_data['mlp_input'] = torch.stack(filtered_data['mlp_input'])
filtered_data['attn_output'] = torch.stack(filtered_data['attn_output'])
filtered_data['residual'] = torch.stack(filtered_data['residual'])
save_dict['cliff'] = filtered_data

print(f"collected {len(save_dict['cliff']['attn_output'])} cliff")

collected 9 cliff
collected 1986 other
collected 9 cliff


In [ ]:
save_dict_path = f"./results/example/attn_output_residual_mlp_pt/creation_layer_info_dict_deepseek-14b_n_{n_gen}_other_{target_layer_idx}.pt"
torch.save(save_dict, save_dict_path)
print(f"Saved creation layer info dict to {save_dict_path}")

Saved creation layer info dict to ./results/example/creation_layer_info_dict_deepseek-14b_n_10_other_25.pt
